Access data from ADLSGen2 through Service Principal
1.  Unset the Accesskey configuration first
2.  Register Service Principal
3.  Each App that is being registred will be given a uniqueapp/client id
4.  create a secret
5.  Configure the Databricks access the storage account via Service Principal

In [0]:
# This will unset the configuration 
storageaccount = "anjanltmstorage"
spark.conf.unset(f"fs.azure.account.key.{storageaccount}.dfs.core.windows.net")

In [0]:
## Fails with the following error as we unset the configuration
## Failure to initialize configuration for storage account anjanltmstorage.dfs.core.windows.net: Invalid configuration value detected for fs.azure.account.key
dbutils.fs.ls("abfss://demo@anjanltmstorage.dfs.core.windows.net")

In [0]:
client_id="Get Client ID"
tenant_id="Get TenanteID"
Secret_value="GEt Secret value"
storageaccount="anjanltmstorage"

In [0]:
# set the configuration using SAS KEY 
## get this pyton code from https://learn.microsoft.com/en-us/azure/databricks/connect/storage/azure-storage 

# service_credential = dbutils.secrets.get(scope="<secret-scope>",key="<service-credential-key>") ## Comment this as we dont need as of now

spark.conf.set(f"fs.azure.account.auth.type.{storageaccount}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storageaccount}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storageaccount}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storageaccount}.dfs.core.windows.net", Secret_value)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storageaccount}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
# Fails with the following error , as we Created SAS key for Ltmsource not for demo container
# Operation failed: "Server failed to authenticate the request
dbutils.fs.ls("abfss://demo@anjanltmstorage.dfs.core.windows.net")

In [0]:
dbutils.fs.ls("abfss://ltmsource@anjanltmstorage.dfs.core.windows.net")

In [0]:
display(dbutils.fs.ls("abfss://ltmsource@anjanltmstorage.dfs.core.windows.net"))

Read Data from ADLS storage file

In [0]:
# just read the data and displays Columns and datatypes
spark.read.csv("abfss://ltmsource@anjanltmstorage.dfs.core.windows.net/circuits.csv")


In [0]:
# dataframe is used to store the read data and display shows the data with Columns C1,C2 ....
df=spark.read.csv("abfss://ltmsource@anjanltmstorage.dfs.core.windows.net/circuits.csv")    
df.show() ## not readable format

In [0]:
#DF - Data frame to store read data with Header option 
df = spark.read.option("header", "true").csv("abfss://ltmsource@anjanltmstorage.dfs.core.windows.net/circuits.csv")
df.display()  ## well readable format